In [ ]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
import warnings; warnings.filterwarnings('ignore')
import pickle
import sys; sys.path.append(os.getcwd())

from fno_LDM.model import FNO
from fno_LDM.train.evaluate import test_reconstruct
from fno_LDM.model import FNO, FNO_GraphVAE, renormalize_weights, load_fno_weights_from_graph, DiT_models, create_diffusion, build_fno_graph_from_structure
from fno_LDM.train.utils import set_cpu_num; set_cpu_num(1)


torch.manual_seed(1)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Pretrained Model

In [ ]:
# Create model:
model_fno = FNO(
    in_channels=2,
    out_channels=2,
    n_modes=(12, 6),
    n_layers=4,
    hidden_channels=64,
)

# --- VAE Hyperparameters ---
internal_dim = 1024      # Common internal dimension (D)
latent_dim = 512        # Latent dimension (h)
num_heads = 8           # Attention heads (internal_dim must be divisible by num_heads)
num_attn_layers = 4     # Renamed from num_gnn_layers
vae = FNO_GraphVAE(
    fno_model=model_fno,
    internal_dim=internal_dim,
    latent_dim=latent_dim,
    num_heads=num_heads,
    num_attn_layers=num_attn_layers,
).to(device)
vae.load_state_dict(torch.load('weights/vae.pt', weights_only=False, map_location=lambda storage, loc: storage)['model'])

#  --- DiT Hyperparameters ---
model = DiT_models['DiT-M'](
    feature_dim=latent_dim,
    layer_num=386,
    cond_dim=2,
    max_cond_period=[1200, 30],
    learn_sigma=False
).to(device)
model.load_state_dict(torch.load(f'weights/ldm.pt', weights_only=False, map_location=lambda storage, loc: storage)["model"])

model.eval()
diffusion = create_diffusion(timestep_respacing="", learn_sigma=False, predict_xstart=False)

# Select a New Environment

In [ ]:
Re, r = [240.0, 19.0] # [210.0, 10.0], [240.0, 19.0], [270.0, 24.0]

# Generate the FNO Weight for the Given Environment

In [ ]:
# Create sampling noise:
z = torch.randn(1, 386, latent_dim, device=device)
c = torch.tensor([[Re, r]], device=device)

# Sample images:
z_samples = diffusion.p_sample_loop(
    model, z.shape, z, clip_denoised=False, model_kwargs=dict(c=c), progress=True, device=device
) # (1, 386, 512)

# decoce
with torch.no_grad():
    dict_samples = vae.decode(z_samples)

# Evaluate the Generated FNO

In [ ]:
# evaluate
fno_graph = build_fno_graph_from_structure(model_fno)
fno_graph['X_lift_node'] = [layer[0] for layer in dict_samples['lift']]
fno_graph['X_block_node'] = [layer[0] for layer in dict_samples['block']]
fno_graph['X_proj_node'] = [layer[0] for layer in dict_samples['proj']]

fno_ = load_fno_weights_from_graph(model_fno, fno_graph)
norm_dict = pickle.load(open(f'zoo/cy_/fno/minmax_dict.pkl', 'rb'))
fno_ = renormalize_weights(fno_, fno_.state_dict(), norm_dict)

o_predictions, g_predictions, targets = test_reconstruct(fno_, cond=[Re, r], device=device)

# Visualize

In [ ]:
vmin, vmax = targets[:, 0].min(), targets[:, 0].max()
for t in range(100):
    plt.figure(figsize=(18, 3))
    plt.subplot(131)
    plt.title(f"Ground Truth (step={ t + 1})")
    plt.imshow(targets[t, 0].T.cpu().numpy(), cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar()
    plt.subplot(132)
    plt.title(f"EnvAd-Diff")
    plt.imshow(g_predictions[t, 0].T.cpu().numpy(), cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar()
    if o_predictions is not None:
        plt.subplot(133)
        plt.title("One-per-Env")
        plt.imshow(o_predictions[t, 0].T.cpu().numpy(), cmap="coolwarm", vmin=vmin, vmax=vmax)
        plt.colorbar()
    plt.show()
    clear_output(wait=True)